# Trabalho de DL - Tradutor (Transformer) - Versão usando Pytorch - G13

### 1. Importando as bibliotecas

In [1]:
import random

from datasets import load_dataset

import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.nn.functional as F

import math

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

### 2. Escolhendo o Device

In [2]:
# Verificar se temos GPU disponível
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Usando: cuda
GPU: Tesla T4


### 3. Hyperparâmetros

In [3]:
# Hyperparametros do trabalho
NUM_SAMPLES = 50000     # Número de amostras a serem usadas para treinamento do tokenizador
BATCH_SIZE = 32         # Tamanho de cada mini batch
#VOCAB_SIZE = 16000     # Tamanho do vocabulário do Tokenizador
MAX_TEXT_LENGTH = 256   # Limita o tamanho máximo do texto
D_MODEL = 128           # Dimensões do modelo
NUM_HEADS = 8           # Número de cabeças do mecanismo de atenção
NUM_ENCODER_LAYERS = 4  # Número de camadas de encoder e decoder
D_FF = 512              # Dimensões do Feed Forward
DROPOUT = 0.1           # Taxa de dropout

### 4. Carregando os dados

In [4]:
dataset = load_dataset(
    "Helsinki-NLP/opus-100",
    "en-pt"
)

README.md:   0%|          | 0.00/65.4k [00:00<?, ?B/s]

en-pt/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  220kB            

en-pt/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

en-pt/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 87.2MB            

en-pt/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

en-pt/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  217kB            

en-pt/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1000000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [5]:
print(dataset)

DatasetDict({
    test: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
    train: Dataset({
        features: ['translation'],
        num_rows: 1000000
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
})


In [6]:
# Um milhão de amostras é muita coisa, reduzindo o treino para 50.000


#train = dataset["train"].select(range(NUM_SAMPLES)) # As 50.000 primeiras

train = dataset["train"].shuffle(seed=13).select(range(NUM_SAMPLES)) # Escolha aleatória

valid = dataset["validation"]

test = dataset["test"]

In [7]:
print(train[0]) # primeira (1)
print(train[-1]) # última (50.000)


{'translation': {'en': 'No, it was just Google.', 'pt': 'Foi só o Google.'}}
{'translation': {'en': 'Hey, T.A., get inside their heads, okay? Yeah?', 'pt': 'Ei, T.A. entra na cabeça deles, ok?'}}


In [8]:
# Escolha 5 amostras da base de treino
for i in random.sample(range(NUM_SAMPLES), 5):

    exemplo = train[i]["translation"]

    print("Português :", exemplo["pt"])
    print("Inglês    :", exemplo["en"])
    print("-"*60)

Português : Vamos!
Inglês    : - Come on!
------------------------------------------------------------
Português : O impacto é, por conseguinte, maciço e é óbvio que o meu país necessita de apoio para poder fazer face a esta grave situação.
Inglês    : Therefore, the impact is massive and it is clear that my country needs support to be able to cope with this serious situation.
------------------------------------------------------------
Português : Por quanto foi vendido o terreno?
Inglês    : What did the property sell for?
------------------------------------------------------------
Português : Deixa-me assumir a investigação, Joseph. Esta é minha casa.
Inglês    : Let me take over the investigation, Joseph.
------------------------------------------------------------
Português : O terrorismo traz perdas incalculáveis, em todos os níveis.
Inglês    : Terrorism brings incalculable losses on all levels.
------------------------------------------------------------


In [9]:
# Separando os idiomas
train_pt = [
    exemplo["translation"]["pt"]
    for exemplo in train
]

train_en = [
    exemplo["translation"]["en"]
    for exemplo in train
]

In [10]:
valid_pt = [
    exemplo["translation"]["pt"]
    for exemplo in valid
]

valid_en = [
    exemplo["translation"]["en"]
    for exemplo in valid
]

test_pt = [
    exemplo["translation"]["pt"]
    for exemplo in test
]

test_en = [
    exemplo["translation"]["en"]
    for exemplo in test
]

In [11]:
print(train_pt[0])
print(train_en[0])

Foi só o Google.
No, it was just Google.


In [12]:
print(f"Treino    : {len(train_pt)}")
print(f"Validação : {len(valid_pt)}")
print(f"Teste     : {len(test_pt)}")

Treino    : 50000
Validação : 2000
Teste     : 2000


### 5. Tokenização

In [13]:
from transformers import AutoTokenizer


In [14]:
# Carrega o BPE pré-treinado do Llama 3
#tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B")
#tokenizer = AutoTokenizer.from_pretrained("NousResearch/Meta-Llama-3-8B-Alternate-Tokenizer")
#tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")
tokenizer = AutoTokenizer.from_pretrained("t5-small")

#VOCAB_SIZE = tokenizer.vocab_size
VOCAB_SIZE = len(tokenizer) # Inclui os tokens especiais
print("Tamanho do vocabulário é :", VOCAB_SIZE)

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

Tamanho do vocabulário é : 32100


In [15]:
# Função que converte a setença em tokens numéricos (ids)
def encoding(sentence):
    return tokenizer.encode(
        sentence,
        truncation=True,
        padding=False,
        add_special_tokens=True,
        max_length=MAX_TEXT_LENGTH)


In [16]:
# Função que converte os ids de volta ao texto
def decoding(ids):
    return tokenizer.decode(ids,
        skip_special_tokens=True)

In [17]:
# Teste
ids_pt = encoding("O aprendizado de máquina é incrível.")
print("PT:", ids_pt)
print(decoding(ids_pt))

PT: [411, 3, 9, 2026, 727, 5584, 26, 32, 20, 3, 51, 2975, 1169, 29, 9, 3, 154, 16, 75, 52, 2, 4911, 5, 1]
O aprendizado de máquina é incrvel.


In [18]:
# Teste
ids_en = encoding("Machine learning is amazing.")
print("EN:", ids_en)
print(decoding(ids_en))

EN: [5879, 1036, 19, 1237, 5, 1]
Machine learning is amazing.


In [19]:
# Retorna uma lista com as strings dos tokens especiais principais
print(tokenizer.all_special_tokens)
# Exemplo de saída: ['<|begin_of_text|>', '<|end_of_text|>', '<|reserved_special_token_0|>', ...]

# Retorna um dicionário mostrando a função de cada um e seu respectivo ID numérico
print(tokenizer.special_tokens_map)
print(tokenizer.all_special_ids)

['</s>', '<unk>', '<pad>', '<extra_id_0>', '<extra_id_1>', '<extra_id_2>', '<extra_id_3>', '<extra_id_4>', '<extra_id_5>', '<extra_id_6>', '<extra_id_7>', '<extra_id_8>', '<extra_id_9>', '<extra_id_10>', '<extra_id_11>', '<extra_id_12>', '<extra_id_13>', '<extra_id_14>', '<extra_id_15>', '<extra_id_16>', '<extra_id_17>', '<extra_id_18>', '<extra_id_19>', '<extra_id_20>', '<extra_id_21>', '<extra_id_22>', '<extra_id_23>', '<extra_id_24>', '<extra_id_25>', '<extra_id_26>', '<extra_id_27>', '<extra_id_28>', '<extra_id_29>', '<extra_id_30>', '<extra_id_31>', '<extra_id_32>', '<extra_id_33>', '<extra_id_34>', '<extra_id_35>', '<extra_id_36>', '<extra_id_37>', '<extra_id_38>', '<extra_id_39>', '<extra_id_40>', '<extra_id_41>', '<extra_id_42>', '<extra_id_43>', '<extra_id_44>', '<extra_id_45>', '<extra_id_46>', '<extra_id_47>', '<extra_id_48>', '<extra_id_49>', '<extra_id_50>', '<extra_id_51>', '<extra_id_52>', '<extra_id_53>', '<extra_id_54>', '<extra_id_55>', '<extra_id_56>', '<extra_id_57>

O tokenizador utilizado foi baseado no t5-small

In [20]:
# bos_id = tokenizer.convert_tokens_to_ids('<|begin_of_text|>')
# print(bos_id)

In [21]:
eos_id = tokenizer.convert_tokens_to_ids('</s>')
print(eos_id)

1


In [22]:
print(tokenizer.pad_token)
pad_id = tokenizer.pad_token_id
print(pad_id)


<pad>
0


In [23]:
# teste de lote
frases = [
    "bom dia",
    "eu gosto de programação",
    "o transformer utiliza atenção",
    "como você está"
]

for frase in frases:
    ids = encoding(frase)

    reconstruida = decoding(ids)

    print("-" * 50)
    print("Original     :", frase)
    print("Reconstruída :", reconstruida)

--------------------------------------------------
Original     : bom dia
Reconstruída : bom dia
--------------------------------------------------
Original     : eu gosto de programação
Reconstruída : eu gosto de programaço
--------------------------------------------------
Original     : o transformer utiliza atenção
Reconstruída : o transformer utiliza atenço
--------------------------------------------------
Original     : como você está
Reconstruída : como você está


In [24]:
frase = "O transformador é eficiente"

#ids = tokenizer.encode(frase_com_eos)
ids = encoding(frase)

print("Frase :", frase)
print("Tokens:", tokenizer.convert_ids_to_tokens(ids))
print("IDs   :", ids)
print("Decode:", decoding(ids))

Frase : O transformador é eficiente
Tokens: ['▁O', '▁transform', 'ador', '▁', 'é', '▁eficient', 'e', '</s>']
IDs   : [411, 3343, 7923, 3, 154, 8539, 15, 1]
Decode: O transformador é eficiente


### 6. Dataset e Dataloader

In [25]:
class TranslationDataset(Dataset):

    def __init__(
        self,
        samples, # Lista de frases em português e inglês
        tokenizer):
        
        self.samples = samples
        self.src_samples, self.tgt_samples = zip(*samples)
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        src_sample = self.src_samples[idx]
        tgt_sample = self.tgt_samples[idx]

        src_ids = (self.tokenizer(src_sample) )
            
        tgt_ids = (self.tokenizer(tgt_sample) )

        return (
            torch.tensor(src_ids, dtype=torch.long),
            torch.tensor(tgt_ids, dtype=torch.long)
        )

In [26]:
# Função que inclui o padding nas amostras
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):

    src_batch, tgt_batch = zip(*batch)

    src_batch = pad_sequence(
        src_batch,
        batch_first=True,
        padding_value=pad_id
    )

    tgt_batch = pad_sequence(
        tgt_batch,
        batch_first=True,
        padding_value=pad_id
    )

    return src_batch, tgt_batch

In [27]:
# Cria o Dataset e o Dataloader de treinamento

from torch.utils.data import DataLoader
train_pairs = list(zip(train_pt, train_en))

train_dataset = TranslationDataset(train_pairs, encoding) # cria o dataset de treinamento

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
) # cria o dataloader de treinamento

In [28]:
len(train_dataset)

50000

In [29]:
len(train_loader)

1563

In [30]:
one_batch_src, one_batch_tgt = next(iter(train_loader))

print(one_batch_src[0])
print(one_batch_tgt[0])

tensor([  679,     3,    32, 18933,     3,    29,     2,    32,   285,     7,
           49,     3,     9,     3,     2,   727,    23,     9,     6,     3,
           15,    76,   361,   509,     3,   287,     3,    15,   521,     5,
            1,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0])
tensor([  156, 18933,   405,    59,   241,     8,     3,     7,  4960,

In [31]:
print(decoding(one_batch_src[0].tolist()))
print(100*'-')
print(decoding(one_batch_tgt[0].tolist()))

Se o Tyler no quiser a ndia, eu fico com ela.
----------------------------------------------------------------------------------------------------
If Tyler does not want the squaw I take the squaw.


In [32]:
# Dataload de validação
valid_pairs = list(zip(valid_pt, valid_en))

valid_dataset = TranslationDataset(
    valid_pairs,
    encoding
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)


In [33]:
train_dataset[0]

(tensor([4452,   23,    3,    7, 4922,    3,   32, 1163,    5,    1]),
 tensor([ 465,    6,   34,   47,  131, 1163,    5,    1]))

In [34]:
src_lengths = [len(item[0]) for item in train_dataset] # obtem os tamanhos das amostras src
tgt_lengths = [len(item[1]) for item in train_dataset] # obtem os tamanhos das amostras tgt

print(max(src_lengths)) # obtem o tamanho da maior amostra src
print(max(tgt_lengths)) # obtem o tamanho da maior amostra tgt

print(sum(src_lengths) / len(src_lengths)) # calcula o tamanho médio das amostras src
print(sum(tgt_lengths) / len(tgt_lengths)) # calcula o tamanho médio das amostras tgt

256
256
27.04088
15.06986


In [35]:
src, tgt = next(iter(train_loader))

print(src.shape)  # (BATCH_SIZE,xx)
print(tgt.shape)  # (BATCH_SIZE,yy)


torch.Size([32, 82])
torch.Size([32, 32])


In [36]:
src[0]

tensor([  71, 3995,  238,  975,   88,   75,   23,  491, 1744,  154,   51,    5,
           1,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0])

In [37]:
print(decoding(src[0].tolist()))
print(decoding(tgt[0].tolist()))


Acho que conheci alguém.
I think I might of met somebody.


### 7. Embedding

In [38]:
import torch
import torch.nn as nn



embedding = nn.Embedding(VOCAB_SIZE, D_MODEL)

# Testando
x = torch.randint(0, VOCAB_SIZE, (32, 20))

emb = embedding(x)

print(emb.shape)


torch.Size([32, 20, 128])


### 8. Positional Encoder

In [39]:
# =====================================================
# Positional Encoding
# =====================================================

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(
            0, max_len, dtype=torch.float
        ).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(
                0, d_model, 2,
                dtype=torch.float
            ) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)

        self.register_buffer("pe", pe)

    def forward(self, x):
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len]


In [40]:
pos = PositionalEncoding(D_MODEL)

x = pos(emb)

### 9. Scaled Dot Product Attention

In [41]:
# =====================================================
# Scaled Dot Product Attention
# =====================================================

class ScaledDotProductAttention(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, q, k, v, mask=None):

        d_k = q.size(-1)

        scores = torch.matmul(
            q, k.transpose(-2, -1)
        ) / math.sqrt(d_k)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        attention = F.softmax(scores, dim=-1)

        output = torch.matmul(attention, v)

        #return output, attention
        return output

### 10. Multi-Head Attention

In [42]:
# =====================================================
# Multi Head Attention
# =====================================================

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)

        self.w_o = nn.Linear(d_model, d_model)

        self.attention = ScaledDotProductAttention()

    def split_heads(self, x):

        batch_size = x.size(0)

        x = x.view(
            batch_size,
            -1,
            self.num_heads,
            self.d_k
        )

        return x.transpose(1, 2)

    def combine_heads(self, x):

        batch_size = x.size(0)

        x = x.transpose(1, 2).contiguous()

        return x.view(
            batch_size,
            -1,
            self.d_model
        )

    def forward(self, q, k, v, mask=None):

        q = self.split_heads(self.w_q(q))
        k = self.split_heads(self.w_k(k))
        v = self.split_heads(self.w_v(v))

        #output, attn = self.attention(
        output = self.attention(
            q,
            k,
            v,
            mask
        )

        output = self.combine_heads(output)

        output = self.w_o(output)

        return output


### 11. Feed Forward Network

In [43]:
# =====================================================
# Feed Forward Network
# =====================================================

class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff=2048, dropout=0.1):
        super().__init__()

        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

### 12. Encoder Layer

In [44]:
# =====================================================
# Encoder Layer
# =====================================================

class EncoderLayer(nn.Module):
    def __init__(
        self,
        d_model,
        num_heads,
        d_ff,
        dropout=0.1
    ):
        super().__init__()

        self.self_attn = MultiHeadAttention(
            d_model,
            num_heads
        )

        self.ffn = PositionwiseFeedForward(
            d_model,
            d_ff,
            dropout
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_mask):

        attn_output = self.self_attn(
            x, x, x, src_mask
        )

        x = self.norm1(
            x + self.dropout(attn_output)
        )

        ff_output = self.ffn(x)

        x = self.norm2(
            x + self.dropout(ff_output)
        )

        return x

### 13. Decoder Layer

In [45]:
# =====================================================
# Decoder Layer
# =====================================================

class DecoderLayer(nn.Module):
    def __init__(
        self,
        d_model,
        num_heads,
        d_ff,
        dropout=0.1
    ):
        super().__init__()

        self.self_attn = MultiHeadAttention(
            d_model,
            num_heads
        )

        self.cross_attn = MultiHeadAttention(
            d_model,
            num_heads
        )

        self.ffn = PositionwiseFeedForward(
            d_model,
            d_ff,
            dropout
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        x,
        encoder_output,
        src_mask,
        tgt_mask
    ):

        attn = self.self_attn(
            x, x, x, tgt_mask
        )

        x = self.norm1(
            x + self.dropout(attn)
        )

        attn = self.cross_attn(
            x,
            encoder_output,
            encoder_output,
            src_mask
        )

        x = self.norm2(
            x + self.dropout(attn)
        )

        ff = self.ffn(x)

        x = self.norm3(
            x + self.dropout(ff)
        )

        return x


### 14. Encoder

In [46]:
# =====================================================
# Encoder
# =====================================================

class Encoder(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model,
        num_layers,
        num_heads,
        d_ff,
        dropout=0.1
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            d_model
        )

        self.pos_encoding = PositionalEncoding(
            d_model
        )

        self.layers = nn.ModuleList(
            [
                EncoderLayer(
                    d_model,
                    num_heads,
                    d_ff,
                    dropout
                )
                for _ in range(num_layers)
            ]
        )

        self.dropout = nn.Dropout(dropout)
        self.d_model = d_model

    def forward(self, src, src_mask):

        x = self.embedding(src) * math.sqrt(
            self.d_model
        )

        x = self.pos_encoding(x)
        x = self.dropout(x)

        for layer in self.layers:
            x = layer(x, src_mask)

        return x


### 15. Decoder

In [47]:
# =====================================================
# Decoder
# =====================================================

class Decoder(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model,
        num_layers,
        num_heads,
        d_ff,
        dropout=0.1
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            d_model
        )

        self.pos_encoding = PositionalEncoding(
            d_model
        )

        self.layers = nn.ModuleList(
            [
                DecoderLayer(
                    d_model,
                    num_heads,
                    d_ff,
                    dropout
                )
                for _ in range(num_layers)
            ]
        )

        self.dropout = nn.Dropout(dropout)
        self.d_model = d_model

    def forward(
        self,
        tgt,
        encoder_output,
        src_mask,
        tgt_mask
    ):

        x = self.embedding(tgt) * math.sqrt(
            self.d_model
        )

        x = self.pos_encoding(x)
        x = self.dropout(x)

        for layer in self.layers:
            x = layer(
                x,
                encoder_output,
                src_mask,
                tgt_mask
            )

        return x

### 16. Transformer

In [48]:
# =====================================================
# Transformer Completo
# =====================================================

class Transformer(nn.Module):
    def __init__(
        self,
        src_vocab_size,
        tgt_vocab_size,
        d_model=512,
        num_layers=6,
        num_heads=8,
        d_ff=2048,
        dropout=0.1
    ):
        super().__init__()

        self.encoder = Encoder(
            src_vocab_size,
            d_model,
            num_layers,
            num_heads,
            d_ff,
            dropout
        )

        self.decoder = Decoder(
            tgt_vocab_size,
            d_model,
            num_layers,
            num_heads,
            d_ff,
            dropout
        )

        self.fc_out = nn.Linear(
            d_model,
            tgt_vocab_size
        )

    def forward(
        self,
        src,
        tgt,
        src_mask,
        tgt_mask
    ):

        encoder_output = self.encoder(
            src,
            src_mask
        )

        decoder_output = self.decoder(
            tgt,
            encoder_output,
            src_mask,
            tgt_mask
        )

        output = self.fc_out(
            decoder_output
        )

        return output


### 17. Máscaras

In [49]:
# =====================================================
# Máscaras
# =====================================================

def create_padding_mask(seq, pad_idx=pad_id):
    return (seq != pad_idx).unsqueeze(1).unsqueeze(2)


def create_causal_mask(size):

    mask = torch.tril(
        torch.ones(size, size)
    )

    return mask.bool().unsqueeze(0).unsqueeze(1)

### 18. Aplicação

In [50]:
src_vocab_size = VOCAB_SIZE
tgt_vocab_size = VOCAB_SIZE

model = Transformer(
    src_vocab_size,
    tgt_vocab_size
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4,
    betas=(0.9, 0.98),
    eps=1e-9
)



In [51]:
#PAD_IDX = tokenizer.token_to_id("[PAD]")

criterion = nn.CrossEntropyLoss(ignore_index=pad_id)


In [52]:
# Testando o modelo

src = torch.randint(
    1,
    src_vocab_size,
    (4, 20)
).to(device)

tgt = torch.randint(
    1,
    tgt_vocab_size,
    (4, 15)
).to(device)

src_mask = create_padding_mask(src).to(device)

tgt_input = tgt[:, :-1].to(device)

tgt_output = tgt[:, 1:].to(device)

tgt_mask = (
    create_padding_mask(tgt_input).to(device)
    & create_causal_mask(tgt_input.size(1)).to(device)
)

output = model(
    src,
    tgt_input,
    src_mask,
    tgt_mask
)

loss = criterion(
    output.reshape(-1, output.size(-1)),
    tgt_output.reshape(-1)
)
print(output.shape)
# (4, 14, VOCAB_SIZE)

torch.Size([4, 14, 32100])


### 19. Loop de treinamento

In [53]:
#device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = Transformer(
    src_vocab_size,
    tgt_vocab_size,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_layers=NUM_ENCODER_LAYERS,
    d_ff=D_FF,
    dropout=DROPOUT
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=pad_id)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4,
    betas=(0.9, 0.98),
    eps=1e-9
)

In [ ]:
import time
import gc
from tqdm.auto import tqdm

def train_epoch(model, dataloader, optimizer, criterion, device, epoch):
    start = time.time()
    model.train()

    total_loss = 0.0
  
   
    for src, tgt in tqdm(dataloader, desc= f'Epoch {epoch+1}'):

        src = src.to(device)
        tgt = tgt.to(device)

        #entrada do decoder
        tgt_input = torch.full(
            (tgt.size(0), tgt.size(1)),
            pad_id,
            device=tgt.device
        )
        tgt_input[:,1:] = tgt[:, :-1]
        
        tgt_output = tgt

        src_mask = create_padding_mask(src).to(device)

        tgt_mask = (
            create_padding_mask(tgt_input)
            &
            create_causal_mask(tgt_input.size(1)).to(device)
        )

        optimizer.zero_grad()

        output = model(src, tgt_input, src_mask, tgt_mask)

        loss = criterion(
            output.reshape(-1, output.size(-1)),
            tgt_output.reshape(-1)
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        total_loss += loss.item()
        # End batchs        
    #liberar memória ao final do treino de uma epoch
    del output
    gc.collect()
    torch.cuda.empty_cache()
    print()          
    print(f'Time taken for 1 epoch: {time.time() - start:.2f} secs')
    return total_loss / len(dataloader)

In [55]:
NUM_EPOCHS = 20

for epoch in range(NUM_EPOCHS):

    loss = train_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        device,
        epoch
    )

    print(f"Epoch {epoch+1:02d} | Loss = {loss:.4f}\n")

Epoch 1:   0%|          | 0/1563 [00:00<?, ?it/s]


Time taken for 1 epoch: 120.49 secs
Epoch 01 | Loss = 6.1773



Epoch 2:   0%|          | 0/1563 [00:00<?, ?it/s]


Time taken for 1 epoch: 125.26 secs
Epoch 02 | Loss = 5.2649



Epoch 3:   0%|          | 0/1563 [00:00<?, ?it/s]

...


Time taken for 1 epoch: 126.06 secs
Epoch 18 | Loss = 4.0686



Epoch 19:   0%|          | 0/1563 [00:00<?, ?it/s]


Time taken for 1 epoch: 125.15 secs
Epoch 19 | Loss = 4.0364



Epoch 20:   0%|          | 0/1563 [00:00<?, ?it/s]


Time taken for 1 epoch: 125.89 secs
Epoch 20 | Loss = 4.0097



### 20. Salvamento do Modelo

In [56]:
torch.save(
    model.state_dict(),
    "translator.pt"
)

In [57]:
# Load
# model.load_state_dict(
#     torch.load("translator.pt")
# )

### 21. Inferência

In [58]:
@torch.no_grad()
def translate(sentence, max_len=None):
    model.eval()

    src_ids = encoding(sentence)

    src = torch.tensor(
        [src_ids],
        dtype=torch.long,
        device=device
    )

    src_mask = create_padding_mask(src)

    encoder_output = model.encoder(src, src_mask)

    tgt = torch.tensor(
        [[pad_id]],
        dtype=torch.long,
        device=device
    )

    if max_len is None:
        max_len = min(len(src_ids) + 30, 128)

    for _ in range(max_len):

        tgt_mask = (
            create_padding_mask(tgt)
            &
            create_causal_mask(tgt.size(1)).to(device)
        )

        decoder_output = model.decoder(
            tgt,
            encoder_output,
            src_mask,
            tgt_mask
        )

        logits = model.fc_out(decoder_output)

        next_token = logits[:, -1].argmax(dim=-1)

        tgt = torch.cat(
            (tgt, next_token.unsqueeze(1)),
            dim=1
        )

        if next_token.item() == eos_id:
            break

    tokens = [
        t
        for t in tgt[0].tolist()
        if t not in (eos_id, pad_id)
    ]

    return decoding(tokens)

In [59]:
sample = random.sample(train_pt, 1)[0]
print(sample)
translate(sample)

Somos do CBI.


"Mom, we're the way."